In [0]:
import time
from pyspark.sql import functions as F

ruta_orders = "/Volumes/workspace/bd26058/experiment/olist_orders_dataset.csv"
ruta_items = "/Volumes/workspace/bd26058/experiment/olist_order_items_dataset.csv"

t0 = time.time()
df_orders = (
    spark.read.option("header", True).option("inferSchema", True).csv(ruta_orders)
)
df_items = (
    spark.read.option("header", True).option("inferSchema", True).csv(ruta_items)
)

cnt_orders = df_orders.count()
cnt_items = df_items.count()
t1 = time.time()

print(f"Pedidos iniciales: {cnt_orders} | Items iniciales: {cnt_items}")
print(f"Tiempo lectura + conteo inicial: {t1 - t0:.2f} s")

Pedidos iniciales: 99441 | Items iniciales: 112650
Tiempo lectura + conteo inicial: 3.48 s


In [0]:
t2 = time.time()

# Filtrar pedidos entregados y unir con items por order_id
df_entregados = (
    df_orders.filter(F.col("order_status") == "delivered")
    .select("order_id", "order_status", "order_purchase_timestamp")
)

df_join = df_entregados.join(df_items, on="order_id", how="inner")

df_resumen = (
    df_join.withColumn("anio", F.year(F.col("order_purchase_timestamp")))
    .groupBy("anio")
    .agg(
        F.count("*").alias("total_items_entregados"),
        F.round(F.sum("price"), 2).alias("venta_total"),
        F.round(F.sum("freight_value"), 2).alias("flete_total")
    )
    .orderBy("anio")
)

total_validos = df_join.count()
t3 = time.time()

print(f"Items en pedidos entregados: {total_validos} (de {cnt_items} iniciales)")
print(f"Tiempo filtro + join + agregacion: {t3 - t2:.2f} s")
display(df_resumen)

Items en pedidos entregados: 110197 (de 112650 iniciales)
Tiempo filtro + join + agregacion: 1.20 s


anio,total_items_entregados,venta_total,flete_total
2016,317,40470.98,6182.76
2017,49556,5962902.01,958633.23
2018,60324,7218125.12,1233459.65
